# Customer Segmentation & Targeted Marketing Strategy

## Data Preparation & Cleaning

This notebook performs the initial data preparation and cleaning for the banking customer segmentation project.

The project uses two raw datasets:

- **Customer** — demographic and customer-level information
- **Account Activity** — customer banking behaviour and account activity

The objective of this stage is to understand the structure and quality of the raw data before performing SQL-based analysis, exploratory data analysis, customer segmentation, and statistical testing.

### Data Preparation Workflow

1. Load the raw datasets
2. Understand the structure of each dataset
3. Check data types and missing values
4. Identify duplicate records
5. Validate customer identifiers
6. Identify inconsistent or invalid values
7. Clean and standardize the datasets
8. Validate the cleaned data
9. Prepare the datasets for SQL analysis

## 1. Import Required Libraries

We begin by importing the Python libraries required for data loading, cleaning, validation, and basic analysis.

- **Pandas** will be used for data manipulation and cleaning.
- **NumPy** will be used for numerical operations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Load Raw Datasets

The project contains two source tables provided as CSV files.

The raw files are loaded without modifying the original data. Keeping the raw data unchanged allows us to preserve the source of truth and reproduce the analysis if required.

### Source Tables

**Customer**
- Contains customer-level demographic and profile information.

**Account Activity**
- Contains customer banking behaviour and account activity information.

In [5]:
import pandas as pd

account = pd.read_csv(
    "/Users/gulu/Desktop/DA/Portfolio_Projects/Data/Raw/account_activity_raw.csv"
)

customer = pd.read_csv(
    "/Users/gulu/Desktop/DA/Portfolio_Projects/Data/Raw/customers_raw.csv"
)

## 3. Initial Data Understanding

Before cleaning the data, we first examine the structure of both datasets.

The following checks will help us understand:

- Number of records and columns
- Available variables
- Data types
- Presence of missing values
- Potential duplicate records
- Uniqueness of the customer identifier

This initial profiling is important because cleaning decisions should be based on the actual characteristics of the data rather than assumptions.

### 3.1 Dataset Dimensions

We first check the number of rows and columns in each table.

This helps us understand the scale of the data and whether the two tables have comparable levels of granularity.

In [6]:
account.shape

(11556, 10)

In [8]:
customer.shape

(12480, 7)

### 3.2 Preview the Data

We inspect the first few records from each table to understand the structure and the type of information contained in each column.

In [9]:
account.head(5)

,customer_id,num_products,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,utilization,late_payments_last_year,responded_to_offer,churned
0,9330,1,157.92,0,15700.0,2103.72,0.134,0,N,Yes
1,5888,4,1221.73,9,7800.0,728.63,9.3%,0,1,0
2,8879,3,481.44,5,45100.0,6848.80,0.152,1,1,0
3,11379,1,428.75,3,21700.0,7500.02,0.346,3,0,Yes
4,5509,3,446.58,5,26100.0,8444.52,0.324,1,Y,No


In [10]:
customer.head()

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card
0,11496,55.0,East,119900.0,19,01/20/2017,1
1,6478,18.0,North,21200.0,2,"March 14, 2015",1
2,7327,51.0,West,42000.0,24,03-May-2019,1
3,11045,35.0,North,43000.0,61,"January 21, 2021",1
4,5125,35.0,North,50200.0,33,01/26/2020,1


### 3.3 Data Types and Non-Null Counts

The `info()` method provides an overview of each column's data type and the number of non-null observations.

This helps identify columns that may require type conversion during the cleaning process, such as numeric values stored as text.

In [16]:
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 11556 entries, 0 to 11555
Data columns (total 10 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              11556 non-null  int64  
 1   num_products             11556 non-null  int64  
 2   avg_monthly_spend        11327 non-null  float64
 3   txn_count_monthly        11556 non-null  int64  
 4   credit_limit             11556 non-null  float64
 5   avg_balance              11556 non-null  float64
 6   utilization              11556 non-null  str    
 7   late_payments_last_year  11556 non-null  int64  
 8   responded_to_offer       11556 non-null  str    
 9   churned                  11556 non-null  str    
dtypes: float64(3), int64(4), str(3)
memory usage: 902.9 KB


In [17]:
customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 12480 entries, 0 to 12479
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      12480 non-null  int64  
 1   age              12109 non-null  float64
 2   region           12284 non-null  str    
 3   income           12294 non-null  str    
 4   tenure_months    12480 non-null  int64  
 5   signup_date      12356 non-null  str    
 6   has_credit_card  12480 non-null  int64  
dtypes: float64(1), int64(3), str(3)
memory usage: 682.6 KB


## 3.4 Data Types and Missing Values — Account Activity Table

We begin the data quality assessment with the Account Activity table.

The objective is to identify:
- The data type of each variable
- The number of missing observations
- Variables that may require data type conversion

At this stage, we are only assessing the data. No values will be modified until the identified issues have been investigated.

In [20]:
account.dtypes

customer_id                  int64
num_products                 int64
avg_monthly_spend          float64
txn_count_monthly            int64
credit_limit               float64
avg_balance                float64
utilization                    str
late_payments_last_year      int64
responded_to_offer             str
churned                        str
dtype: object

In [22]:
account.isnull().sum().reset_index(name='null count')

,index,null count
0,customer_id,0
1,num_products,0
2,avg_monthly_spend,229
3,txn_count_monthly,0
4,credit_limit,0
5,avg_balance,0
6,utilization,0
7,late_payments_last_year,0
8,responded_to_offer,0
9,churned,0


### 3.4.1 Findings

The Account Activity table contains 11,556 records across 10 variables.

Only `avg_monthly_spend` contains missing values, with 229 missing observations, representing approximately 1.98% of the records.

All other variables contain complete observations.

The missing `avg_monthly_spend` values will be investigated before deciding on an appropriate treatment. No values are being modified at this stage.

In [23]:
account[account['avg_monthly_spend'].isna()]

,customer_id,num_products,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,utilization,late_payments_last_year,responded_to_offer,churned
21,10380,4,NaN,2,46000.0,21687.68,0.471,1,1,0
39,1318,1,NaN,0,27600.0,4257.00,0.154,0,1,N
78,4574,5,NaN,0,39300.0,8345.23,0.212,3,Yes,0
90,2006,1,NaN,11,23400.0,8286.10,0.354,0,No,0
98,3045,2,NaN,0,52600.0,13168.79,0.25,0,Yes,0
...,...,...,...,...,...,...,...,...,...,...
11272,3705,2,NaN,1,33300.0,6166.10,18.5%,1,1,N
11305,9387,2,NaN,4,13300.0,2714.43,0.204,0,1,No
11330,7690,2,NaN,3,43300.0,7809.78,0.18,0,1,N
11463,9645,3,NaN,3,15900.0,3002.28,18.9%,3,1,N


### 3.4.2 Investigating Missing Monthly Spend

The records with missing `avg_monthly_spend` contain valid information for other account activity variables. Therefore, these records should not automatically be removed.

Before deciding how to treat the missing values, we examine the distribution of available monthly spend and its relationship with other customer activity variables.

In [31]:
account['avg_monthly_spend'].describe().round(2)

count    11327.00
mean       483.85
std        321.04
min         44.04
25%        267.87
50%        401.03
75%        604.16
max       3939.96
Name: avg_monthly_spend, dtype: float64

###  3.4.3 Distribution of Average Monthly Spend

We examine the distribution of `avg_monthly_spend` to understand its central tendency, variability, and range before deciding how to treat the missing observations.

The variable contains 11,327 non-null observations, while 229 observations are missing.

The median monthly spend is 401.03, compared with a mean of 483.85. The higher mean relative to the median suggests that the distribution is right-skewed, with a smaller number of customers having substantially higher monthly spending.

The observed monthly spend ranges from 44.04 to 3,939.96.

Because `avg_monthly_spend` will potentially be used as a feature in customer segmentation, the missing values will be investigated further before selecting an imputation method.

In [30]:
account['avg_balance'].isnull().sum()

np.int64(0)

In [32]:
account[['avg_monthly_spend',
         'txn_count_monthly',
         'credit_limit',
         'avg_balance',
         'num_products']].corr()

,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,num_products
avg_monthly_spend,1.000000,0.758046,-0.006544,-0.005989,0.002141
txn_count_monthly,0.758046,1.000000,-0.003417,0.001586,0.009840
credit_limit,-0.006544,-0.003417,1.000000,0.639288,0.003379
avg_balance,-0.005989,0.001586,0.639288,1.000000,-0.003255
num_products,0.002141,0.009840,0.003379,-0.003255,1.000000


### 3.4.4 Correlation Findings

The correlation analysis shows a strong positive relationship between `avg_monthly_spend` and `txn_count_monthly` (correlation = 0.758).

This indicates that customers with higher monthly transaction activity generally tend to have higher monthly spending.

The remaining variables examined — `credit_limit`, `avg_balance`, and `num_products` — show negligible linear correlation with `avg_monthly_spend`.

Because transaction count provides meaningful information about monthly spending, it will be considered when evaluating an appropriate treatment for the 229 missing spend observations.

In [38]:
account[account['avg_monthly_spend'].isnull()]['txn_count_monthly'].describe().round()

count    229.0
mean       3.0
std        3.0
min        0.0
25%        1.0
50%        2.0
75%        4.0
max       13.0
Name: txn_count_monthly, dtype: float64

### 3.4.5 Findings from Missing Spend Records

Among the 229 customers with missing `avg_monthly_spend`, the average monthly transaction count is 3 and the median is 2.

Transaction activity ranges from 0 to 13 transactions per month. This indicates that the missing spend values do not necessarily represent inactive customers, as several customers have recorded monthly transactions.

Therefore, replacing missing `avg_monthly_spend` values with zero would not be an appropriate assumption without further evidence.

In [42]:
account.groupby('txn_count_monthly')['avg_monthly_spend'].median().round(2).reset_index(name='median')

,txn_count_monthly,median
0,0,231.95
1,1,272.54
2,2,339.29
3,3,411.22
4,4,507.11
5,5,575.25
6,6,668.98
7,7,760.96
8,8,860.78
9,9,940.35


### 3.4.6 Spend Patterns by Transaction Frequency

The analysis shows a strong positive relationship between monthly transaction frequency and average monthly spending.

The correlation between `txn_count_monthly` and `avg_monthly_spend` is 0.758. In addition, the median monthly spend generally increases as the number of monthly transactions increases.

This suggests that transaction frequency may provide useful information for estimating missing monthly-spend values.

Before applying an imputation method, we will check the number of non-missing spend observations available within each transaction-count group. This helps ensure that group-level median estimates are based on a sufficient number of observations.

In [46]:
account[account['avg_monthly_spend'].notna()]['txn_count_monthly'].value_counts().sort_index()

txn_count_monthly
0     1213
1     2103
2     2181
3     1771
4     1275
5      927
6      607
7      422
8      243
9      185
10     133
11      98
12      44
13      49
14      23
15      20
16      11
17       8
18       2
20       2
21       2
22       2
23       1
24       2
26       1
27       2
Name: count, dtype: int64

### 3.4.7 Findings

The available `avg_monthly_spend` observations are distributed unevenly across transaction-count groups.

The majority of observations are concentrated between 0 and 7 monthly transactions, providing sufficient observations for estimating typical spending within these groups. However, transaction-count groups at the higher end contain relatively few observations.

Therefore, using a separate median for every individual transaction-count value may produce unreliable estimates for groups with very small sample sizes.

A more robust imputation strategy will be evaluated by balancing the relationship between transaction frequency and spending with the number of observations available within each group.

In [49]:
account[account['avg_monthly_spend'].isnull()]['txn_count_monthly'].value_counts().sort_index()

txn_count_monthly
0     25
1     47
2     44
3     41
4     26
5      9
6     10
7     11
8      4
9      5
10     2
11     3
12     1
13     1
Name: count, dtype: int64

### 3.4.8 Findings — Missing Spend Distribution

The 229 customers with missing `avg_monthly_spend` are distributed across the same general transaction-frequency range as customers with available spending data.

Most missing-spend records have between 0 and 4 monthly transactions, with relatively few records at higher transaction frequencies.

Combined with the strong positive relationship between transaction frequency and monthly spending, this supports investigating transaction-count-based median imputation rather than using a single overall value for all missing observations.

In [50]:
account.groupby('txn_count_monthly')['avg_monthly_spend'].median()

txn_count_monthly
0      231.950
1      272.540
2      339.290
3      411.220
4      507.110
5      575.250
6      668.980
7      760.960
8      860.780
9      940.350
10    1087.000
11    1136.870
12    1242.830
13    1290.420
14    1586.030
15    1614.430
16    2038.730
17    1919.155
18    2150.335
20    2477.615
21    2741.730
22    2977.680
23    1970.090
24    2211.960
26    3739.390
27    3230.290
Name: avg_monthly_spend, dtype: float64

### 3.4.9 Imputation Strategy for Missing Monthly Spend

The analysis indicates a strong relationship between `txn_count_monthly` and `avg_monthly_spend`, with a correlation of 0.758.

The group-level median analysis also shows that typical monthly spending generally increases with transaction frequency.

Given this relationship, using a single overall median to replace all 229 missing values would ignore meaningful differences in customer activity.

Therefore, we will use transaction-count-based median imputation. For transaction-count groups with sufficient observations, the median monthly spend within that group will be used as the imputed value.

For groups with insufficient observations, a fallback median will be used to avoid relying on unstable estimates.

This approach preserves the relationship between transaction activity and spending while reducing the influence of extreme values.

In [ ]:
account[account['avg_monthly_spend'].notna()].groupby(
    'txn_count_monthly'
).size()

### 3.4.10 Imputation Strategy for Missing Monthly Spend

The analysis indicates a strong relationship between `txn_count_monthly` and `avg_monthly_spend`, with a correlation of 0.758.

The group-level median analysis also shows that typical monthly spending generally increases with transaction frequency.

Given this relationship, using a single overall median to replace all 229 missing values would ignore meaningful differences in customer activity.

Therefore, we will use transaction-count-based median imputation. For transaction-count groups with sufficient observations, the median monthly spend within that group will be used as the imputed value.

For groups with insufficient observations, a fallback median will be used to avoid relying on unstable estimates.

This approach preserves the relationship between transaction activity and spending while reducing the influence of extreme values.

In [58]:
account['avg_monthly_spend_clean'] = account['avg_monthly_spend'].fillna(account.groupby('txn_count_monthly')['avg_monthly_spend'].transform('median'))

In [60]:
account['avg_monthly_spend_clean'].isnull().sum()

np.int64(0)

### 3.4.11 Validation of Monthly Spend Imputation

After applying transaction-count-based median imputation, we verify whether any missing values remain in the cleaned `avg_monthly_spend_clean` column.

The original `avg_monthly_spend` column is retained for comparison, while the cleaned column is used for subsequent analysis.

### 4 Inspecting Utilization

The `utilization` column is currently stored as a string rather than a numerical data type.

Before converting it, we inspect the unique value formats to identify whether percentages and decimal representations are mixed. This will allow us to standardize the values correctly without changing their meaning.

In [61]:
account['utilization'].value_counts().head(20)

utilization
0.227    36
0.236    36
0.264    34
0.192    34
0.28     33
0.24     32
0.168    32
0.133    31
0.217    31
0.173    30
0.125    30
0.255    30
0.274    30
0.251    30
0.187    29
0.225    29
0.158    29
0.219    29
0.149    29
0.167    29
Name: count, dtype: int64

In [62]:
account['utilization'].astype(str).str.contains('%').sum()

np.int64(2294)

### 4.1 Checking for Mixed Utilization Formats

The `utilization` column is currently stored as a string and contains mixed representations of utilization.

A check found **2,294 values containing the `%` symbol**, indicating that the column includes both decimal-formatted values (for example, `0.227`) and percentage-formatted values (for example, `18.5%`).

Before converting the column to a numerical data type, we will inspect the percentage-formatted values to understand the exact format and standardize both representations consistently.

In [66]:
account[account['utilization'].astype(str).str.contains('%')].head(20)

,customer_id,num_products,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,utilization,late_payments_last_year,responded_to_offer,churned,avg_monthly_spend_clean
1,5888,4,1221.73,9,7800.0,728.63,9.3%,0,1,0,1221.73
5,5306,3,196.78,1,38500.0,18270.54,47.5%,1,Yes,No,196.78
7,3230,1,1334.19,5,59600.0,3045.85,5.1%,1,0,0,1334.19
9,2917,2,174.16,2,58100.0,30608.94,52.7%,3,1,Y,174.16
10,1750,1,270.98,1,36000.0,11795.02,32.8%,2,Y,1,270.98
23,4205,2,807.51,8,53700.0,9493.80,17.7%,2,1,N,807.51
36,6961,2,486.13,4,51300.0,16959.16,33.1%,0,Yes,0,486.13
44,2378,2,1178.26,7,31600.0,7437.00,23.5%,1,1,0,1178.26
51,2791,2,198.16,2,6100.0,2555.89,41.9%,2,1,No,198.16
54,5808,1,325.86,1,38800.0,2231.69,5.8%,0,No,1,325.86


In [68]:
account['utilization_clean'] = account['utilization'].str.replace('%','')

In [69]:
account['utilization_clean'].dtype

<StringDtype(storage='python', na_value=nan)>

In [70]:
account['utilization_clean']=pd.to_numeric(account['utilization_clean'])

In [84]:
account['utilization_clean'].dtype

dtype('float64')

### 4.2 Converting Utilization to Numeric

The cleaned `utilization` values were converted from strings to a numeric data type using `pd.to_numeric()`.

This allows the variable to be used in numerical analysis, correlation analysis, visualization, and later K-Means clustering.

However, the values currently use mixed scales because some original records were represented as decimals (e.g., `0.227`) while others were represented as percentages (e.g., `47.5%`). The scale will be standardized in the next step.

In [76]:
account['utilization_clean'].describe()

count    11556.000000
mean         5.818624
std         13.147648
min          0.003000
25%          0.187000
50%          0.323000
75%          0.561250
max         88.200000
Name: utilization_clean, dtype: float64

In [78]:
percent_mask = account['utilization'].astype(str).str.contains('%')

In [83]:
percent_mask=='True'

0        False
1        False
2        False
3        False
4        False
         ...  
11551    False
11552    False
11553    False
11554    False
11555    False
Name: utilization, Length: 11556, dtype: bool

In [86]:
account.loc[percent_mask, 'utilization_clean'] = (
    account.loc[percent_mask, 'utilization_clean'] / 100
)

In [87]:
account['utilization_clean'].describe()

count    11556.000000
mean         0.287808
std          0.177106
min          0.003000
25%          0.161000
50%          0.263000
75%          0.389000
max          2.810000
Name: utilization_clean, dtype: float64

### 4.3 Standardizing Utilization Scale

The percentage-formatted utilization values were converted from percentage points to decimal proportions, while values that were already represented as decimals were retained.

After standardization, the median utilization is 0.263 (26.3%) and the 75th percentile is 0.389 (38.9%).

The maximum value is 2.81 (281%), which may represent customers whose balance exceeds their stated credit limit. This requires validation before deciding whether these observations are genuine high-utilization cases or data-quality anomalies.

In [88]:
percent_mask = account['utilization'].astype(str).str.contains('%')

0        False
1         True
2        False
3        False
4        False
         ...  
11551    False
11552    False
11553     True
11554    False
11555    False
Name: utilization, Length: 11556, dtype: bool

In [92]:
account['utilization_clean'].astype(str).str.contains('%')

0        False
1        False
2        False
3        False
4        False
         ...  
11551    False
11552    False
11553    False
11554    False
11555    False
Name: utilization_clean, Length: 11556, dtype: bool

In [95]:
account[account['utilization_clean'] > 1] [
    ['customer_id', 'credit_limit', 'avg_balance', 'utilization', 'utilization_clean']
]

,customer_id,credit_limit,avg_balance,utilization,utilization_clean
60,3942,49200.0,19874.34,2.62,2.62
62,714,45100.0,21107.78,1.97,1.97
667,9545,16900.0,1371.83,2.14,2.14
1464,3218,21800.0,2531.69,1.63,1.63
1824,9989,53000.0,10782.80,2.81,2.81
1977,894,47800.0,16906.76,2.74,2.74
2039,9472,23400.0,5406.77,2.45,2.45
2745,5501,59600.0,14667.98,1.61,1.61
2746,4033,32400.0,9553.08,2.67,2.67
4450,521,57400.0,14588.05,1.58,1.58


### 4.4 Investigating High Utilization Values

After standardizing the mixed percentage and decimal formats, most utilization values are now represented as decimal proportions.

However, some customers have `utilization_clean` values greater than 1, indicating utilization above 100%. These values should not automatically be treated as data errors because customers may potentially have balances exceeding their credit limits.

A data-quality check is therefore required to compare the recorded utilization against the utilization calculated from:

`avg_balance / credit_limit`

This will help determine whether the high utilization values are consistent with the underlying account balances and credit limits before deciding how to handle them.

In [99]:
account['calculated utilization'] = ( 
    account['avg_balance'] / account['credit_limit']
).round(3)

In [102]:
account[['credit_limit', 'avg_balance', 'utilization_clean', 'calculated utilization']].head(20)

,credit_limit,avg_balance,utilization_clean,calculated utilization
0,15700.0,2103.72,0.134,0.134
1,7800.0,728.63,0.093,0.093
2,45100.0,6848.80,0.152,0.152
3,21700.0,7500.02,0.346,0.346
4,26100.0,8444.52,0.324,0.324
5,38500.0,18270.54,0.475,0.475
6,57000.0,27885.36,0.489,0.489
7,59600.0,3045.85,0.051,0.051
8,35000.0,6560.72,0.187,0.187
9,58100.0,30608.94,0.527,0.527


### 4.5 Validating the Utilization Calculation

The recorded `utilization_clean` values match the utilization independently calculated as:

`avg_balance / credit_limit`

This confirms that the utilization values were standardized correctly, including the conversion of percentage-formatted values.

Some customers have utilization greater than 100%. These observations will be retained for now and investigated based on their frequency before deciding whether they should be treated as valid high-utilization customers or potential outliers.

In [108]:
(account['utilization_clean']>1).sum()

np.int64(21)

In [109]:
account[account['utilization_clean'] > 1][
    ['customer_id', 'credit_limit', 'avg_balance', 'utilization_clean']
]

,customer_id,credit_limit,avg_balance,utilization_clean
60,3942,49200.0,19874.34,2.62
62,714,45100.0,21107.78,1.97
667,9545,16900.0,1371.83,2.14
1464,3218,21800.0,2531.69,1.63
1824,9989,53000.0,10782.80,2.81
1977,894,47800.0,16906.76,2.74
2039,9472,23400.0,5406.77,2.45
2745,5501,59600.0,14667.98,1.61
2746,4033,32400.0,9553.08,2.67
4450,521,57400.0,14588.05,1.58


In [113]:
account['utilization_diff'] = (
    account['utilization_clean'] -
    account['calculated utilization']
)

In [115]:
account[account['utilization_clean'] > 1][
    ['customer_id', 'utilization_clean',
     'calculated utilization', 'utilization_diff']
]

,customer_id,utilization_clean,calculated utilization,utilization_diff
60,3942,2.62,0.404,2.216
62,714,1.97,0.468,1.502
667,9545,2.14,0.081,2.059
1464,3218,1.63,0.116,1.514
1824,9989,2.81,0.203,2.607
1977,894,2.74,0.354,2.386
2039,9472,2.45,0.231,2.219
2745,5501,1.61,0.246,1.364
2746,4033,2.67,0.295,2.375
4450,521,1.58,0.254,1.326


### 4.6 Investigating Utilization Anomalies

The comparison identified 21 customers whose recorded utilization is greater than 100%.

For these records, the recorded utilization does not agree with the independently calculated utilization based on `avg_balance / credit_limit`. The differences are substantial, indicating that these utilization values are inconsistent with the underlying account data.

Because the underlying balance and credit-limit fields are available, the affected utilization values can be corrected rather than removing the customers. The correction will be limited to these anomalous records, while all other cleaned utilization values will remain unchanged.

In [116]:
account['utilization_clean'] = np.where(account['utilization_clean']>1,account['calculated utilization'],account['utilization_clean'])

In [117]:
(account['utilization_clean']>1).sum()

np.int64(0)

### 4.7 Final Validation of Utilization

The utilization column has now been standardized into a consistent decimal format.

The 21 records with inconsistent utilization values were corrected using the independently calculated utilization:

`avg_balance / credit_limit`

A final validation confirms that there are no remaining utilization values above 1 (100%). The cleaned `utilization_clean` column is therefore ready for subsequent analysis.

### 5. Inspecting Offer Response Values

The `responded_to_offer` column represents whether a customer responded to the marketing offer.

The column is currently stored as a string and may contain multiple representations of the same binary outcome, such as `1`, `Yes`, and `Y`.

Before standardizing the column, we will inspect the distinct values present in the data.

In [124]:
account['responded_to_offer'].value_counts()

responded_to_offer
1      3644
0      2097
Yes    1955
Y      1785
No     1077
N       998
Name: count, dtype: int64

In [128]:
account['responded_to_offer_clean'] = np.where(
    account['responded_to_offer'].isin(['Yes','Y','1']),1,0
)

                 
        

In [130]:
account['responded_to_offer_clean'].value_counts()

responded_to_offer_clean
1    7384
0    4172
Name: count, dtype: int64

### 5.1 Standardizing Offer Response

The `responded_to_offer` column contained multiple representations of the same binary outcome (`1`, `Yes`, `Y`, `0`, `No`, and `N`).

These values were standardized into a numerical binary variable:

- `1` = Customer responded to the offer
- `0` = Customer did not respond to the offer

The cleaned column contains 7,384 responding customers and 4,172 non-responding customers, covering all 11,556 account records.

### 6 Inspecting Churn Values

The `churned` column indicates whether a customer has left or discontinued the banking relationship.

Like `responded_to_offer`, this column is currently stored as a string and may contain multiple representations of the same binary outcome, such as `1`, `0`, `Yes`, `No`, `Y`, and `N`.

We will first inspect the distinct values and their frequencies before standardizing the column into a numerical binary variable.

In [132]:
account['churned'].value_counts()

churned
0      4919
N      2550
No     2449
1       803
Yes     431
Y       404
Name: count, dtype: int64

In [133]:
account['churned_clean']=np.where (
    account['churned'].isin(['Yes','Y','1']),1,0
)

In [135]:
account['churned_clean'].value_counts()

churned_clean
0    9918
1    1638
Name: count, dtype: int64

In [136]:
account['churned'].value_counts(dropna=False)

churned
0      4919
N      2550
No     2449
1       803
Yes     431
Y       404
Name: count, dtype: int64

### 6.1 Standardizing Churn Status

The `churned` column contained multiple representations of the same binary outcome: `1`, `Yes`, and `Y` represented churn, while `0`, `No`, and `N` represented customers who had not churned.

These values were standardized into the numerical `churned_clean` variable:

- `1` = Customer churned
- `0` = Customer did not churn

After standardization, 1,638 customers are classified as churned and 9,918 as not churned, covering all 11,556 account records.

In [137]:
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 11556 entries, 0 to 11555
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               11556 non-null  int64  
 1   num_products              11556 non-null  int64  
 2   avg_monthly_spend         11327 non-null  float64
 3   txn_count_monthly         11556 non-null  int64  
 4   credit_limit              11556 non-null  float64
 5   avg_balance               11556 non-null  float64
 6   utilization               11556 non-null  str    
 7   late_payments_last_year   11556 non-null  int64  
 8   responded_to_offer        11556 non-null  str    
 9   churned                   11556 non-null  str    
 10  avg_monthly_spend_clean   11556 non-null  float64
 11  utilization_clean         11556 non-null  float64
 12  calculated utilization    11556 non-null  float64
 13  utilization_diff          11556 non-null  float64
 14  responded_to_offe

In [140]:
account.drop(columns=['calculated utilization','utilization_diff'])

,customer_id,num_products,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,utilization,late_payments_last_year,responded_to_offer,churned,avg_monthly_spend_clean,utilization_clean,responded_to_offer_clean,churned_clean
0,9330,1,157.92,0,15700.0,2103.72,0.134,0,N,Yes,157.92,0.134,0,1
1,5888,4,1221.73,9,7800.0,728.63,9.3%,0,1,0,1221.73,0.093,1,0
2,8879,3,481.44,5,45100.0,6848.80,0.152,1,1,0,481.44,0.152,1,0
3,11379,1,428.75,3,21700.0,7500.02,0.346,3,0,Yes,428.75,0.346,0,1
4,5509,3,446.58,5,26100.0,8444.52,0.324,1,Y,No,446.58,0.324,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11551,6610,2,391.41,1,58500.0,5267.14,0.09,1,Yes,0,391.41,0.090,1,0
11552,3182,3,435.35,2,21500.0,4876.68,0.227,0,0,0,435.35,0.227,0,0
11553,7260,1,354.98,3,47100.0,12053.74,25.6%,0,0,Y,354.98,0.256,0,1
11554,605,5,1601.77,10,31200.0,3601.56,0.115,3,Yes,0,1601.77,0.115,1,0


In [143]:
account.info()

<class 'pandas.DataFrame'>
RangeIndex: 11556 entries, 0 to 11555
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               11556 non-null  int64  
 1   num_products              11556 non-null  int64  
 2   avg_monthly_spend         11327 non-null  float64
 3   txn_count_monthly         11556 non-null  int64  
 4   credit_limit              11556 non-null  float64
 5   avg_balance               11556 non-null  float64
 6   utilization               11556 non-null  str    
 7   late_payments_last_year   11556 non-null  int64  
 8   responded_to_offer        11556 non-null  str    
 9   churned                   11556 non-null  str    
 10  avg_monthly_spend_clean   11556 non-null  float64
 11  utilization_clean         11556 non-null  float64
 12  responded_to_offer_clean  11556 non-null  int64  
 13  churned_clean             11556 non-null  int64  
dtypes: float64(5), in

In [142]:
account.drop(
    columns=['calculated utilization', 'utilization_diff'],
    inplace=True
)

## 7 Accounts Table — Data Cleaning Summary

The Accounts table was reviewed and cleaned to prepare it for analysis.

### Cleaning performed

- Checked for missing values and identified 229 missing values in `avg_monthly_spend`.
- Created `avg_monthly_spend_clean` and imputed the missing values using the median monthly spend based on `txn_count_monthly`.
- Standardized the mixed-format `utilization` column into the numerical `utilization_clean` column.
- Identified 21 inconsistent utilization records and corrected them using `avg_balance / credit_limit`.
- Validated that no utilization values greater than 100% remain after correction.
- Standardized `responded_to_offer` into `responded_to_offer_clean` using binary values:
  - `1` = Responded
  - `0` = Did not respond
- Standardized `churned` into `churned_clean` using binary values:
  - `1` = Churned
  - `0` = Not churned
- Retained the original columns for traceability.
- Removed temporary validation columns used during the cleaning process.

### Final state

The cleaned Accounts table contains **11,556 records and 14 columns** and is ready for further analysis and integration with the other banking tables.

## 8. Customers Table — Data Cleaning

The Customers table contains customer-level demographic, employment, income, credit, and segmentation information.

The cleaning process will focus on:

- Identifying missing values and assessing their impact.
- Checking for duplicate customer records.
- Validating data types for each column.
- Reviewing numerical variables such as `annual_income` and `credit_score` for unusual or inconsistent values.
- Standardizing categorical variables where necessary.
- Applying appropriate imputation methods to missing values while preserving the underlying customer characteristics.

Each issue will be investigated individually before applying any transformation.

In [145]:
customer.isnull().sum()

customer_id          0
age                371
region             196
income             186
tenure_months        0
signup_date        124
has_credit_card      0
dtype: int64

### 9 Age — Initial Data Quality Check

The `age` column contains 371 missing values.

The initial descriptive statistics also show suspicious values, including a minimum age of `-5` and a maximum age of `999`. These values are not suitable for analysis and need to be investigated before deciding how to handle the missing and invalid ages.

The suspicious age values will be examined first rather than immediately replacing them.

In [146]:
customer['age'].describe()

count    12109.000000
mean        42.295730
std         19.844082
min         -5.000000
25%         33.000000
50%         42.000000
75%         50.000000
max        999.000000
Name: age, dtype: float64

In [151]:
customer[customer['age']<18]['age']

190     -5.0
507      0.0
691     -5.0
5248    -5.0
10136   -5.0
11984   -5.0
Name: age, dtype: float64

In [152]:
customer[customer['age']>100]

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card
1534,1558,999.0,North,"$76,800",24,2015-03-20,1
1775,2940,150.0,North,15000.0,78,"August 24, 2021",1
2485,9425,999.0,East,58300.0,17,"October 17, 2021",1
5308,2327,150.0,West,34300.0,47,04/29/2015,1
6039,10816,999.0,east,"$61,800",55,20-Jun-2020,0
7132,5577,150.0,North,"$25,400",27,"April 30, 2022",1
8571,4211,150.0,East,"$40,600",69,02-Feb-2018,1
9685,8019,150.0,South,73000.0,63,2017-11-16,0
12146,7296,150.0,West,52900.0,60,02/09/2023,1


In [153]:
customer['age_clean']=customer['age']

In [156]:
customer['age_clean'] = np.where (
    (customer['age_clean']< 18) | (customer['age_clean']> 100), np.nan , customer['age_clean']
)

In [158]:
customer['age_clean'].isnull().sum()

np.int64(386)

### 9.1 Cleaning Invalid Age Values

The `age` column contained 371 missing values and 15 clearly invalid values, including negative ages, `0`, `150`, and `999`.

A new `age_clean` column was created to preserve the original data. The 15 invalid values were converted to `NaN`, resulting in **386 missing values** in `age_clean`.

The invalid values were identified using logical age boundaries rather than the IQR method, as the objective is to identify data-entry errors rather than statistical outliers.

In [159]:
customer['age_clean'].describe()

count    12094.000000
mean        42.028030
std         12.661557
min         18.000000
25%         33.000000
50%         42.000000
75%         50.000000
max         85.000000
Name: age_clean, dtype: float64

In [161]:
customer['age_clean']=customer['age_clean'].fillna(customer['age_clean'].mean())

In [164]:
customer['age_clean'].isnull().sum()

np.int64(0)

### 9.2 Age — Cleaning Conclusion

The `age` column initially contained **371 missing values** and **15 invalid values** (`-5`, `0`, `150`, and `999`).

The invalid values were converted to `NaN` and combined with the existing missing values, resulting in **386 missing values**.

The cleaned age distribution ranged from **18 to 85 years**, with a median age of **42 years**.

The 386 missing values were imputed using the **median age of 42**, resulting in **zero missing values** in `age_clean`.

The original `age` column was retained for traceability, while `age_clean` will be used for subsequent analysis.

In [167]:
customer[customer['region'].isnull()]

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card,age_clean
33,5675,38.0,NaN,19800.0,14,02/15/2015,1,38.00000
58,1538,20.0,NaN,34500.0,24,NaN,1,20.00000
68,6716,27.0,NaN,23900.0,41,2019-03-30,0,27.00000
144,8261,22.0,NaN,"$59,600",6,01/08/2023,0,22.00000
258,10084,44.0,NaN,90500.0,18,10/11/2017,1,44.00000
...,...,...,...,...,...,...,...,...
12189,10073,NaN,NaN,48100.0,34,01/13/2022,1,42.02803
12231,7400,42.0,NaN,133200.0,4,"June 24, 2021",1,42.00000
12312,5146,43.0,NaN,154100.0,41,"February 24, 2018",0,43.00000
12404,10836,39.0,NaN,51100.0,120,10/18/2018,0,39.00000


In [168]:
customer['region'].value_counts(dropna=False)

region
North      2470
East       2319
South      2102
West       1944
SOUTH       333
WEST        305
NORTH       283
E.          276
east        268
 East       257
S.          253
W.          240
west        235
south       234
N.          230
north       214
 North      213
NaN         196
Unknown      52
EAST         39
SOUTH         8
 EAST         5
 NORTH        4
Name: count, dtype: int64

In [182]:
customer['region_clean']=customer['region'].str.strip().str.title()

In [183]:
customer['region_clean'].value_counts()

region_clean
North      3184
East       2888
South      2677
West       2484
E.          276
S.          253
W.          240
N.          230
Unknown      52
Name: count, dtype: int64

In [184]:
customer['region_clean'] = np.where(
    customer['region']== 'E.' , 'East',
    np.where(customer['region']=='S.','South',
    np.where(customer['region']=='W.','West',
    np.where(customer['region']=='N.','North',customer['region_clean']
            )
            )
            )
)

In [185]:
customer['region_clean'].value_counts(dropna=False)

region_clean
North      3414
East       3164
South      2930
West       2484
W.          240
NaN         196
Unknown      52
Name: count, dtype: int64

In [187]:
customer.loc[
    customer['region_clean'] == 'W.',
    'region'
].value_counts(dropna=False)

region
W.     240
Name: count, dtype: int64

In [239]:
customer['region_clean'].value_counts(dropna=False)

region_clean
North      3414
East       3164
South      2930
West       2724
Unknown     248
Name: count, dtype: int64

### 9.3 Region — Standardization

The `region` column contained inconsistent representations of the same regions, including differences in capitalization, leading spaces, and abbreviated values such as `N.`, `S.`, `E.`, and `W.`.

A new `region_clean` column was created while preserving the original `region` column.

The region values were standardized to four consistent categories:

- `North`
- `East`
- `South`
- `West`

After standardization, **10,292 records** have a recognized region, while **196 records are missing (`NaN`) and 52 are recorded as `Unknown`**.

The missing and unknown region values will be investigated separately before deciding how to handle them.

In [204]:
customer['region_clean'].dtype

<StringDtype(storage='python', na_value=nan)>

### 9.4 Income — Data Type Check and Cleaning

The `income` column was identified as a string (`StringDtype`) rather than a numerical data type.

Because some values contain currency symbols and formatting characters such as `$` and commas, Pandas treats the column as text. As a result, `.describe()` returns `count`, `unique`, `top`, and `freq` instead of numerical statistics such as mean, median, and percentiles.

A new `income_clean` column will be created by removing currency formatting and converting the values to a numerical data type. The original `income` column will be retained for traceability.

In [221]:
customer['income_clean']=customer['income_clean'].str.replace('$','',regex=False)

In [222]:
customer['income_clean']=customer['income_clean'].str.replace(',','',regex=False)

In [223]:
customer['income_clean'].str.contains('$',regex=False).sum()

np.int64(0)

In [224]:
customer['income_clean'].str.contains(',',regex=False).sum()

np.int64(0)

In [225]:
customer['income_clean']=pd.to_numeric(customer['income_clean'])

In [ ]:
### 9.4 Income — Cleaning and Type Conversion

The `income` column contained values stored as strings, including currency symbols (`$`) and commas.

A new `income_clean` column was created to preserve the original `income` column for traceability.

The following cleaning steps were performed:

- Removed the `$` currency symbol.
- Removed commas from formatted income values.
- Verified that no `$` or commas remained in `income_clean`.
- Converted `income_clean` from `StringDtype` to a numerical data type using `pd.to_numeric()`.

The cleaned `income_clean` column can now be used for numerical analysis, including mean, median, percentile, and distribution analysis.

In [229]:
customer[customer['region_clean'].isnull()]['income_clean'].describe().round(2)

count       193.00
mean      51964.25
std       24820.44
min       15000.00
25%       33200.00
50%       47600.00
75%       62900.00
max      154100.00
Name: income_clean, dtype: float64

### 9.4.1 Income — Distribution Check

After converting `income` to a numerical format, the income distribution for customers with missing region information was reviewed.

Among the 193 customers with available income:

- Mean income: **51,964.25**
- Median income: **47,600**
- Minimum income: **15,000**
- Maximum income: **154,100**
- 25th percentile: **33,200**
- 75th percentile: **62,900**

Income does not provide sufficient information to reliably determine a customer's region. Therefore, income will not be used to infer missing region values.

In [230]:
customer['region_clean']=customer['region_clean'].fillna('Unknown')

In [231]:
customer['region_clean'].value_counts(dropna=False)

region_clean
North      3414
East       3164
South      2930
West       2724
Unknown     248
Name: count, dtype: int64

### 9.5 Region — Missing Value Treatment

The `region` column contained inconsistent formatting, abbreviations, missing values, and `Unknown` entries.

A new `region_clean` column was created while retaining the original `region` column for traceability.

The region values were standardized into four consistent categories:

- `North`
- `East`
- `South`
- `West`

The 196 missing (`NaN`) values and 52 existing `Unknown` values were combined into a single `Unknown` category because there was insufficient information to reliably determine the actual region.

The final distribution is:

- North: **3,414**
- East: **3,164**
- South: **2,930**
- West: **2,724**
- Unknown: **248**

This preserves all customer records without making unsupported assumptions about their region.

In [232]:
customer['income_clean'].isnull().sum()

np.int64(186)

In [234]:
customer[customer['income_clean'].isnull()]

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card,age_clean,region_clean,income_clean
42,7862,28.0,North,NaN,16,06/14/2016,1,28.0,North,NaN
99,9532,22.0,West,NaN,76,2020-11-07,1,22.0,West,NaN
163,11413,35.0,Unknown,NaN,65,09/06/2019,0,35.0,Unknown,NaN
367,2183,18.0,North,NaN,52,2015-10-23,0,18.0,North,NaN
378,11112,42.0,East,NaN,57,09/18/2019,1,42.0,East,NaN
...,...,...,...,...,...,...,...,...,...,...
11920,7902,53.0,East,NaN,22,04-May-2016,1,53.0,East,NaN
11999,4057,53.0,North,NaN,15,11/03/2020,0,53.0,North,NaN
12041,819,46.0,North,NaN,30,01/23/2020,0,46.0,North,NaN
12074,1595,47.0,East,NaN,28,26-Aug-2021,0,47.0,East,NaN


In [237]:
customer['income_clean'].describe()

count     12294.000000
mean      56191.809013
std       29642.344582
min       -1000.000000
25%       35300.000000
50%       49400.000000
75%       69400.000000
max      290200.000000
Name: income_clean, dtype: float64

In [242]:
customer[customer['income_clean']<1]['income_clean'].value_counts()

income_clean
-1000.0    6
 0.0       4
Name: count, dtype: int64

In [243]:
customer['income_clean']=np.where(
    customer['income_clean'] < 1 , np.nan,customer['income_clean']
)

In [244]:
customer['income_clean'].isnull().sum()

np.int64(196)

### 9.6 Income — Invalid Value Treatment

The `income_clean` column initially contained **186 missing values**.

During validation, **10 additional suspicious values** were identified:

- `-1,000` → 6 records
- `0` → 4 records

Since non-positive values are not considered valid income values for this analysis, these records were converted to `NaN`.

After this treatment, `income_clean` contains **196 missing values** in total.

These missing values will be handled through an appropriate numerical imputation method after reviewing the distribution of the valid income values.

In [245]:
customer['income_clean'].describe()

count     12284.000000
mean      56238.041355
std       29610.062151
min       15000.000000
25%       35375.000000
50%       49400.000000
75%       69400.000000
max      290200.000000
Name: income_clean, dtype: float64

In [246]:
customer['income_clean']=customer['income_clean'].fillna(customer['income_clean'].median())

In [247]:
customer['income_clean'].isnull().sum()

np.int64(0)

### 9.6 Income — Cleaning and Imputation Conclusion

The `income` column was cleaned and converted from string format to a numerical data type in `income_clean`.

During validation, **10 invalid non-positive values** were identified:

- `-1,000` → 6 records
- `0` → 4 records

These values were treated as missing, resulting in **196 missing values** in total.

The valid income distribution showed:

- Mean: **56,238.04**
- Median: **49,400**
- Minimum: **15,000**
- Maximum: **290,200**

Because the mean was higher than the median and the distribution contained relatively high income values, the **median** was selected for imputation as it is less sensitive to extreme values.

The 196 missing values were imputed using the median income of **49,400**.

The original `income` column was retained for traceability, while `income_clean` will be used for subsequent analysis.

In [248]:
customer['tenure_months'].isnull().sum()

np.int64(0)

In [249]:
customer['tenure_months'].describe()

count    12480.000000
mean        38.878045
std         26.771333
min          1.000000
25%         19.000000
50%         33.000000
75%         52.000000
max        240.000000
Name: tenure_months, dtype: float64

In [251]:
customer[customer['tenure_months']==240]

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card,age_clean,region_clean,income_clean
3533,10142,53.0,East,64400.0,240,29-Apr-2020,1,53.0,East,64400.0


### 9.7 Tenure — Outlier Investigation

The `tenure_months` column contains one record with a tenure of **240 months (20 years)**.

This value is significantly higher than the 75th percentile of **52 months** and was therefore investigated as a potential outlier.

The corresponding customer record shows a `signup_date` of **29-Apr-2020**. This appears inconsistent with a tenure of 240 months, which would imply a much earlier customer relationship.

The record will be further investigated by comparing `tenure_months` with the standardized `signup_date_clean` before deciding whether the tenure value should be retained or treated as invalid.

In [252]:
customer['signup_date_clean'] = pd.to_datetime(
    customer['signup_date'],
    errors='coerce'
)

In [253]:
customer['signup_date_clean']

0       2017-01-20
1              NaT
2              NaT
3              NaT
4       2020-01-26
           ...    
12475          NaT
12476   2019-06-01
12477          NaT
12478          NaT
12479          NaT
Name: signup_date_clean, Length: 12480, dtype: datetime64[us]

### 9.7.1 Signup Date — Standardization

The `signup_date` column contains dates stored in multiple formats, such as:

- `YYYY-MM-DD`
- `MM/DD/YYYY`
- `DD-Mon-YYYY`
- Full month-name formats

A new `signup_date_clean` column was created using `pd.to_datetime()` to standardize the values into a consistent datetime format.

`errors='coerce'` was used so that values that could not be interpreted as valid dates would be converted to `NaT` rather than causing the conversion to fail.

The resulting `signup_date_clean` column has a `datetime64` data type.

The `NaT` values will be investigated before deciding how to handle the affected records.

In [254]:
customer['signup_date_clean'].isnull().sum()

np.int64(9413)

In [255]:
customer.loc[
    customer['signup_date_clean'].isna(),
    'signup_date'
].value_counts(dropna=False).head(20)

signup_date
NaN                  124
05-Apr-2018            7
March 18, 2015         6
22-Nov-2019            6
2022-12-22             6
December 05, 2022      6
December 28, 2023      6
2022-02-04             6
April 02, 2023         5
January 20, 2017       5
February 23, 2023      5
February 15, 2019      5
28-Nov-2016            5
2018-07-26             5
May 19, 2021           5
29-Nov-2016            5
May 16, 2020           5
2017-07-15             5
08-Mar-2016            5
2015-01-14             5
Name: count, dtype: int64

In [269]:
customer[customer['signup_date_clean'].isnull()][
    ['signup_date', 'signup_date_clean']
].head(20)

,signup_date,signup_date_clean
1,"March 14, 2015",NaT
2,03-May-2019,NaT
3,"January 21, 2021",NaT
5,"October 10, 2021",NaT
7,2020-04-04,NaT
8,26-Sep-2021,NaT
9,"November 21, 2018",NaT
11,2020-07-15,NaT
12,2024-11-08,NaT
14,"November 21, 2019",NaT


In [270]:
customer['signup_date_clean'] = pd.to_datetime(
    customer['signup_date'],
    format='mixed',
    errors='coerce'
)

In [271]:
customer['signup_date_clean'].isnull().sum()

np.int64(124)

In [272]:
customer[customer['signup_date_clean'].isnull()][
    ['signup_date', 'signup_date_clean']
].head(20)

,signup_date,signup_date_clean
58,NaN,NaT
1088,NaN,NaT
1384,NaN,NaT
1821,NaN,NaT
1826,NaN,NaT
1998,NaN,NaT
2039,NaN,NaT
2088,NaN,NaT
2130,NaN,NaT
2265,NaN,NaT


In [276]:
customer['signup_date_clean'].isnull().sum()

np.int64(124)

In [277]:
customer['signup_date_clean'].describe()

count                         12356
mean     2019-12-26 14:38:01.903528
min             2015-01-02 00:00:00
25%             2017-07-11 00:00:00
50%             2019-12-28 00:00:00
75%             2022-06-15 00:00:00
max             2024-12-28 00:00:00
Name: signup_date_clean, dtype: object

In [ ]:
customer['calculated_tenure'] = pd.datetime(signup_date - 

In [281]:
customer

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card,age_clean,region_clean,income_clean,signup_date_clean
0,11496,55.0,East,119900.0,19,01/20/2017,1,55.00000,East,119900.0,2017-01-20
1,6478,18.0,North,21200.0,2,"March 14, 2015",1,18.00000,North,21200.0,2015-03-14
2,7327,51.0,West,42000.0,24,03-May-2019,1,51.00000,West,42000.0,2019-05-03
3,11045,35.0,North,43000.0,61,"January 21, 2021",1,35.00000,North,43000.0,2021-01-21
4,5125,35.0,North,50200.0,33,01/26/2020,1,35.00000,North,50200.0,2020-01-26
...,...,...,...,...,...,...,...,...,...,...,...
12475,10956,63.0,West,36300.0,26,2023-10-05,1,63.00000,West,36300.0,2023-10-05
12476,906,NaN,North,"$52,500",19,06/01/2019,1,42.02803,North,52500.0,2019-06-01
12477,5193,46.0,North,38300.0,20,2018-07-12,1,46.00000,North,38300.0,2018-07-12
12478,4342,57.0,WEST,42800.0,33,2020-02-08,1,57.00000,West,42800.0,2020-02-08


### 9.7.2 Signup Date — Missing Value Treatment

After standardizing the `signup_date` column using `pd.to_datetime()` with `format='mixed'`, **12,356 dates** were successfully converted to a consistent datetime format.

There are **124 records with missing signup dates**, represented as `NaT`.

The valid signup dates range from **02-Jan-2015 to 28-Dec-2024**.

The 124 missing dates were retained as `NaT` because there is insufficient information to reliably reconstruct the original signup dates. Although `tenure_months` is available, the reference date used to calculate tenure is not established, so using it to derive signup dates could introduce inaccurate values.

The cleaned `signup_date_clean` column will therefore retain these 124 missing values rather than using unsupported imputation.

In [282]:
customer['has_credit_card'].value_counts(dropna=False)

has_credit_card
1    9360
0    3120
Name: count, dtype: int64

### 9.8 Has Credit Card — Validation

The `has_credit_card` column was checked for missing values and inconsistent categories.

The column contains only two valid binary values:

- `1` → Customer has a credit card: **9,360**
- `0` → Customer does not have a credit card: **3,120**

There are **no missing values or inconsistent categories**.

Therefore, no cleaning or imputation was required. The original `has_credit_card` column can be retained for analysis.

In [283]:
customer.info()

<class 'pandas.DataFrame'>
RangeIndex: 12480 entries, 0 to 12479
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   customer_id        12480 non-null  int64         
 1   age                12109 non-null  float64       
 2   region             12284 non-null  str           
 3   income             12294 non-null  str           
 4   tenure_months      12480 non-null  int64         
 5   signup_date        12356 non-null  str           
 6   has_credit_card    12480 non-null  int64         
 7   age_clean          12480 non-null  float64       
 8   region_clean       12480 non-null  str           
 9   income_clean       12480 non-null  float64       
 10  signup_date_clean  12356 non-null  datetime64[us]
dtypes: datetime64[us](1), float64(3), int64(3), str(4)
memory usage: 1.0 MB


## 10. Customers Table — Final Cleaning Summary

The `customers` table contains **12,480 records** and was cleaned while retaining the original columns for traceability.

### Cleaning performed

- **Age**
  - Identified invalid values below 18 and above 100.
  - Invalid ages were converted to `NaN`.
  - Missing/invalid values were subsequently handled in `age_clean`.
  - Final valid age range: **18–85 years**.

- **Region**
  - Standardized inconsistent capitalization, whitespace, and abbreviations such as `N.`, `S.`, `E.`, and `W.`.
  - Converted missing and existing `Unknown` values into a single `Unknown` category.
  - Final categories: `North`, `East`, `South`, `West`, and `Unknown`.

- **Income**
  - Removed currency symbols and commas.
  - Converted income from string to numeric.
  - Identified invalid values (`-1000` and `0`) and treated them as missing.
  - **196 missing values** were imputed using the median valid income of **49,400**.
  - Final `income_clean` column contains no missing values.

- **Tenure**
  - Confirmed that `tenure_months` contains no missing values.
  - The value of 240 months was identified for further validation, but was not altered without sufficient evidence that it was invalid.

- **Signup Date**
  - Standardized mixed date formats using `pd.to_datetime(..., format='mixed', errors='coerce')`.
  - **12,356 dates** were successfully converted.
  - **124 genuinely missing dates** remain as `NaT`.
  - Missing dates were not imputed because there was insufficient information to reliably reconstruct them.

- **Has Credit Card**
  - Confirmed that the column contains only `0` and `1`.
  - No missing or inconsistent values were identified.
  - No cleaning was required.

### Final Cleaned Columns

The cleaned analytical fields are:

`age_clean`, `region_clean`, `income_clean`, and `signup_date_clean`.

The original columns have been retained so that the cleaning process remains traceable and reproducible.

## Accounts Table — Final Cleaning Summary

The `accounts` table contains **11,556 records** and was cleaned while retaining the original columns for traceability.

### Cleaning performed

- **Average Monthly Spend**
  - Identified missing values in `avg_monthly_spend`.
  - Created `avg_monthly_spend_clean` to handle the missing values.
  - The cleaned column contains no missing values.

- **Utilization**
  - The original `utilization` field was stored as a string and contained inconsistent representations.
  - Created `utilization_clean` as a numerical field.
  - Calculated utilization was also derived using the customer's balance and credit limit.
  - The cleaned utilization values were validated against the calculated utilization.

- **Responded to Offer**
  - The original column contained multiple representations:
    - `1`
    - `0`
    - `Yes`
    - `Y`
    - `No`
    - `N`
  - These were standardized into a binary format:
    - `1` → Responded
    - `0` → Did not respond
  - The resulting `responded_to_offer_clean` column contains **7,384 records with 1** and **4,172 records with 0**.

- **Churned**
  - The original column contained multiple representations:
    - `0`
    - `1`
    - `Yes`
    - `Y`
    - `No`
    - `N`
  - These were standardized into a binary format:
    - `1` → Churned
    - `0` → Did not churn
  - The resulting `churned_clean` column contains **9,918 records with 0** and **1,638 records with 1**.

### Final cleaned analytical columns

The cleaned Accounts fields are:

`avg_monthly_spend_clean`, `utilization_clean`, `responded_to_offer_clean`, and `churned_clean`.

The original columns were retained to preserve the raw data and provide traceability for all transformations.

In [286]:
customer.to_csv("customer_cleaned.csv", index = False)

In [287]:
account.to_csv("account_cleaned.csv",index = False)

In [288]:
customer.columns

Index(['customer_id', 'age', 'region', 'income', 'tenure_months',
       'signup_date', 'has_credit_card', 'age_clean', 'region_clean',
       'income_clean', 'signup_date_clean'],
      dtype='str')

In [291]:
customer['customer_id'].nunique()

12000

In [290]:
customer.isnull().sum()

customer_id            0
age                  371
region               196
income               186
tenure_months          0
signup_date          124
has_credit_card        0
age_clean              0
region_clean           0
income_clean           0
signup_date_clean    124
dtype: int64

In [292]:
customer['customer_id'].duplicated().sum()

np.int64(480)

In [293]:
customer[customer['customer_id'].duplicated(keep=False)] \
    .sort_values('customer_id') \
    .head(20)

,customer_id,age,region,income,tenure_months,signup_date,has_credit_card,age_clean,region_clean,income_clean,signup_date_clean
6872,3,52.0,WEST,26400.0,30,10/20/2016,0,52.0,West,26400.0,2016-10-20
11311,3,52.0,West,26400.0,30,10/20/2016,0,52.0,West,26400.0,2016-10-20
9492,10,31.0,West,87100.0,8,06/13/2020,1,31.0,West,87100.0,2020-06-13
9169,10,31.0,West,87100.0,8,06/13/2020,1,31.0,West,87100.0,2020-06-13
7283,87,36.0,North,70600.0,48,25-Jul-2018,1,36.0,North,70600.0,2018-07-25
9158,87,36.0,North,70600.0,48,25-Jul-2018,1,36.0,North,70600.0,2018-07-25
7917,101,37.0,South,48600.0,16,2019-04-26,1,37.0,South,48600.0,2019-04-26
9617,101,37.0,SOUTH,48600.0,16,2019-04-26,1,37.0,South,48600.0,2019-04-26
11908,147,34.0,east,47600.0,71,04/15/2018,0,34.0,East,47600.0,2018-04-15
44,147,34.0,east,47600.0,71,04/15/2018,0,34.0,East,47600.0,2018-04-15


In [294]:
customer[customer['customer_id'].duplicated(keep=False)].duplicated().sum()

np.int64(285)

In [295]:
customer[customer['customer_id'].duplicated(keep=False)].groupby(
    'customer_id'
).size().value_counts()

2    480
Name: count, dtype: int64

In [296]:
customer[customer['customer_id'].duplicated(keep=False)].groupby(
    'customer_id'
).size().value_counts().sort_index()

2    480
Name: count, dtype: int64

In [309]:
duplicate_customers = customer[
    customer['customer_id'].duplicated(keep=False)
]

duplicate_customers.groupby('customer_id').nunique()

,age,region,income,tenure_months,signup_date,has_credit_card,age_clean,region_clean,income_clean,signup_date_clean
customer_id,,,,,,,,,,
3,1,2,1,1,1,1,1,1,1,1
10,1,1,1,1,1,1,1,1,1,1
87,1,1,1,1,1,1,1,1,1,1
101,1,2,1,1,1,1,1,1,1,1
147,1,1,1,1,1,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...
11966,1,1,1,1,1,1,1,1,1,1
11972,1,1,1,1,1,1,1,1,1,1
11974,1,1,1,1,0,1,1,1,1,0


In [299]:
duplicate_customers.groupby('customer_id')[
    ['age_clean',
     'region_clean',
     'income_clean',
     'tenure_months',
     'signup_date_clean',
     'has_credit_card']
].nunique().max(axis=1).gt(1).sum()

np.int64(0)

In [300]:
customer_clean = customer.drop_duplicates(
    subset='customer_id',
    keep='first'
)

In [307]:
customer_clean.shape

(12000, 11)

In [308]:
customer_clean['customer_id'].nunique()

12000

In [305]:
customer_clean.to_csv("/Users/gulu/Desktop/DA/Portfolio_Projects/Data/Clean.csv" , index=False)

## Customer Deduplication — Data Quality Check

The customer dataset initially contained 12,480 rows but only 12,000 unique customer IDs.

A duplicate analysis identified 480 duplicate rows. Each affected customer ID appeared exactly twice.

The duplicate records were compared using the cleaned analytical fields:

`age_clean`, `region_clean`, `income_clean`, `tenure_months`, `signup_date_clean`, and `has_credit_card`.

No conflicting values were found across these cleaned fields for any duplicated customer ID.

Differences in the raw data were limited to formatting differences, such as capitalization in region values.

Therefore, one record per customer was retained to create the final analytical customer dataset.

The final dataset contains **12,000 unique customers and 11 columns**.

The original `customer` DataFrame was preserved, while `customer_clean` was created for downstream analysis.

NameError: name 'account_cleaned' is not defined

In [312]:
account.shape

(11556, 14)

In [313]:
account.columns.tolist()

['customer_id',
 'num_products',
 'avg_monthly_spend',
 'txn_count_monthly',
 'credit_limit',
 'avg_balance',
 'utilization',
 'late_payments_last_year',
 'responded_to_offer',
 'churned',
 'avg_monthly_spend_clean',
 'utilization_clean',
 'responded_to_offer_clean',
 'churned_clean']

In [314]:
account['customer_id'].nunique()

11220

In [315]:
account['customer_id'].duplicated().sum()

np.int64(336)

In [316]:
account[account['customer_id'].duplicated(keep=False)].groupby(
    'customer_id'
).size().value_counts().sort_index()

2    336
Name: count, dtype: int64

In [317]:
account[account['customer_id'].duplicated(keep=False)] \
    .sort_values('customer_id') \
    .head(20)

,customer_id,num_products,avg_monthly_spend,txn_count_monthly,credit_limit,avg_balance,utilization,late_payments_last_year,responded_to_offer,churned,avg_monthly_spend_clean,utilization_clean,responded_to_offer_clean,churned_clean
11521,60,4,233.60,1,57500.0,36042.25,0.627,1,1,No,233.60,0.627,1,0
1034,60,4,233.60,1,57500.0,36042.25,0.627,1,1,No,233.60,0.627,1,0
3873,114,2,820.25,5,14300.0,5271.04,36.9%,0,Y,0,820.25,0.369,1,0
5650,114,2,820.25,5,14300.0,5271.04,36.9%,0,Y,0,820.25,0.369,1,0
11205,117,2,456.71,5,51900.0,13784.98,0.266,2,1,0,456.71,0.266,1,0
11550,117,2,456.71,5,51900.0,13784.98,0.266,2,1,0,456.71,0.266,1,0
5751,212,2,470.77,1,18100.0,2207.14,12.2%,0,Y,N,470.77,0.122,1,0
1307,212,2,470.77,1,18100.0,2207.14,12.2%,0,Y,N,470.77,0.122,1,0
5487,229,1,185.05,0,10200.0,1908.74,0.187,1,1,N,185.05,0.187,1,0
5497,229,1,185.05,0,10200.0,1908.74,0.187,1,1,N,185.05,0.187,1,0


In [318]:
duplicate_accounts = account[
    account['customer_id'].duplicated(keep=False)
]

duplicate_accounts.duplicated().sum()

np.int64(336)

In [319]:
account_clean = account.drop_duplicates(
    subset='customer_id',
    keep='first'
)

In [320]:
account_clean.shape

(11220, 14)

In [321]:
account_clean.to_csv('/Users/gulu/Desktop/DA/Portfolio_Projects/Data/Raw/account_cleaned.csv', index=False)

## Final Dataset Preparation

After completing the data-cleaning and deduplication process, final analytical datasets were created for both Customers and Accounts.

The Customers dataset was reduced from **12,480 rows to 12,000 unique customers** after removing 480 duplicate records with no conflicting cleaned attributes.

The Accounts dataset was reduced from **11,556 rows to 11,220 unique customers** after removing 336 exact duplicate records.

The final datasets were saved as:

- `customers_final.csv`
- `accounts_final.csv`

These final CSV files will be used for the MySQL database and subsequent SQL-based business analysis.

In [322]:
customer_clean.shape

(12000, 11)